In [1]:
# BAN6800 – BA-05: Data Preparation and Missing-Value Preprocessing
# CGSL Predictive Maintenance Project

import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# Load the UCI APS training and test datasets from GitHub

train_url = "https://raw.githubusercontent.com/dvfekorigha-create/BAN6800-cgsl-predictive-maintenance/main/data/raw/aps_failure_training_set.csv"
test_url = "https://raw.githubusercontent.com/dvfekorigha-create/BAN6800-cgsl-predictive-maintenance/main/data/raw/aps_failure_test_set.csv"

df_train = pd.read_csv(
    train_url,
    skiprows=20,
    na_values="na"
)

df_test = pd.read_csv(
    test_url,
    skiprows=20,
    na_values="na"
)

print("Training shape:", df_train.shape)
print("Test shape:", df_test.shape)
print("Training target distribution:")
print(df_train["class"].value_counts())

Training shape: (60000, 171)
Test shape: (16000, 171)
Training target distribution:
class
neg    59000
pos     1000
Name: count, dtype: int64


In [3]:
# Identify features with more than 80% missing values using training data only

missing_rate = df_train.drop(columns=["class"]).isna().mean()

high_missing_features = missing_rate[missing_rate > 0.80].index.tolist()

print("Features with more than 80% missing values:")
print(high_missing_features)

print("\nNumber of features to remove:", len(high_missing_features))

# Apply the same feature removal to both datasets
df_train_reduced = df_train.drop(columns=high_missing_features)
df_test_reduced = df_test.drop(columns=high_missing_features)

print("\nOriginal training shape:", df_train.shape)
print("Reduced training shape:", df_train_reduced.shape)

print("Original test shape:", df_test.shape)
print("Reduced test shape:", df_test_reduced.shape)

Features with more than 80% missing values:
['bq_000', 'br_000']

Number of features to remove: 2

Original training shape: (60000, 171)
Reduced training shape: (60000, 169)
Original test shape: (16000, 171)
Reduced test shape: (16000, 169)


In [4]:
# Separate target and features

X_train = df_train_reduced.drop(columns=["class"])
y_train = df_train_reduced["class"]

X_test = df_test_reduced.drop(columns=["class"])
y_test = df_test_reduced["class"]

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)

Training features: (60000, 168)
Test features: (16000, 168)


In [5]:
from sklearn.impute import SimpleImputer

# Fit the imputer ONLY on the training features
imputer = SimpleImputer(
    strategy="median",
    add_indicator=True
)

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

print("Imputation completed successfully.")
print("Transformed training shape:", X_train_imputed.shape)
print("Transformed test shape:", X_test_imputed.shape)
print("Remaining missing values in training:",
      np.isnan(X_train_imputed).sum())
print("Remaining missing values in test:",
      np.isnan(X_test_imputed).sum())

Imputation completed successfully.
Transformed training shape: (60000, 335)
Transformed test shape: (16000, 335)
Remaining missing values in training: 0
Remaining missing values in test: 0


In [6]:
# Verify that training and test feature structures are consistent

print("Training transformed shape:", X_train_imputed.shape)
print("Test transformed shape:", X_test_imputed.shape)

print("\nFeature count difference:",
      X_train_imputed.shape[1] - X_test_imputed.shape[1])

print("\nTraining target shape:", y_train.shape)
print("Test target shape:", y_test.shape)

Training transformed shape: (60000, 335)
Test transformed shape: (16000, 335)

Feature count difference: 0

Training target shape: (60000,)
Test target shape: (16000,)


## BA-05 Missing-Value Preprocessing Decisions

The preprocessing strategy was developed from the BA-04 data-quality findings.

### 1. Removal of extremely sparse features

Two training-set features exceeded 80% missingness:

- `bq_000`
- `br_000`

These features were removed from both the training and test datasets to maintain an identical feature structure.

### 2. Missing-value imputation

The remaining numeric features were median-imputed.

The imputation model was fitted using the training features only and then applied unchanged to the test features. This prevents information from the test dataset from influencing preprocessing parameters and reduces the risk of data leakage.

### 3. Missingness indicators

Missingness indicators were retained using `add_indicator=True`. This preserves information about whether an observation was originally missing, which may provide useful predictive information.

### 4. Result

After preprocessing:

- Training dataset: 60,000 observations × 335 transformed features
- Test dataset: 16,000 observations × 335 transformed features
- Remaining missing values in training: 0
- Remaining missing values in test: 0

The transformed feature structure is identical between training and test datasets.

### 5. Limitation

The anonymized UCI feature names do not provide sufficient domain information to determine the physical meaning of individual measurements. Therefore, the 80% missingness threshold and median-imputation approach are treated as an initial analytical strategy and will be evaluated further during EDA and modelling.

In [7]:
# Save preprocessing outputs for reproducibility

np.save("X_train_imputed.npy", X_train_imputed)
np.save("X_test_imputed.npy", X_test_imputed)

y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Preprocessed datasets saved successfully.")
print("- X_train_imputed.npy")
print("- X_test_imputed.npy")
print("- y_train.csv")
print("- y_test.csv")

Preprocessed datasets saved successfully.
- X_train_imputed.npy
- X_test_imputed.npy
- y_train.csv
- y_test.csv


In [8]:
from google.colab import files

files.download("X_train_imputed.npy")
files.download("X_test_imputed.npy")
files.download("y_train.csv")
files.download("y_test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>